# Solution 2.2.4 — Transforming Data & Creating New Features

### Path Setup

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_CLEANED_DIR = '../../data/10_cleaned'
FILE_NAME = 'datania_households_clean.csv'
clean_path = os.path.join(DATA_CLEANED_DIR, FILE_NAME)

df = pd.read_csv(clean_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded:', df.shape)
df.head()

---

## Task 1 — Recode: per-capita, bands, and label maps

In [ ]:
df['income_per_capita'] = df['income_dkw'] / df['hh_size']

df[['hh_id', 'income_dkw', 'hh_size', 'income_per_capita']].head()

In [ ]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 65, 120],
    labels=['Child', 'Adult', 'Elderly']
)
df['age_group'].value_counts(dropna=False)

In [ ]:
# Map education codes to labels
education_map = {1: 'Primary', 2: 'Secondary', 3: 'Tertiary', 4: 'Higher levels'}
df['education_label'] = df['education_code'].map(education_map)

df['education_label'].value_counts(dropna=False)

**Questions:**

- `map()` returns `NaN` for any code not in the dictionary. Which households end up with a missing `education_label`, and why (think back to 2.2.3)?
- An income of `NaN` divided by `hh_size` gives what? Check `income_per_capita` for households whose income was missing.

**Answers:**

- The household whose `education_code` was the sentinel `99` (recoded to `NaN` in 2.2.3) has no entry in `education_map`, so `map()` returns `NaN`. Always re-check categories after mapping.
- `NaN / hh_size` is `NaN`: households with missing income keep a missing per-capita value, which is the honest result.

---

## Task 2 — Conditional assignment with `np.where()` and `.loc[]`

In [ ]:
df['area_type'] = np.where(df['pop_density'] > 500, 'Urban', 'Rural')

df[['hh_id', 'pop_density', 'urban_rural', 'area_type']].head(8)

In [ ]:
df['hh_category'] = 'Standard'
df.loc[df['hh_size'] >= 7, 'hh_category'] = 'Large'
df.loc[df['hh_size'] <= 2, 'hh_category'] = 'Small'

df[['hh_id', 'hh_size', 'hh_category']].head(10)

**Questions:**

- Where the reported `urban_rural` and the density-based `area_type` disagree, which would you trust, and how would you investigate?
- `np.where()` treats a `NaN` population density as "not > 500" and labels it `Rural`. Is that the behaviour you want? How could you make the unknown explicit?

**Answers:**

- When the reported `urban_rural` and the density-based `area_type` disagree, the reported value usually reflects an official classification while density is a proxy — investigate by checking the enumerator notes and the actual density threshold rather than assuming either is "correct".
- `np.where()` labels a `NaN` density as `Rural` because `NaN > 500` is `False`. If "unknown" should stay unknown, build the column with `np.select()` and add a condition `df['pop_density'].isna()` -> `'Unknown'`.

---

## Task 3 — Clean column creation with `assign()`

In [ ]:
df = df.assign(
    income_thousands = df['income_dkw'] / 1000,
    pop_density_log = np.log(df['pop_density'])
)
df[['hh_id', 'income_dkw', 'income_thousands', 'pop_density', 'pop_density_log']].head()

In [ ]:
df = df.assign(
    income_per_capita = lambda x: x['income_dkw'] / x['hh_size'],
    high_income = lambda x: np.where(x['income_per_capita'] > 25000, 'Yes', 'No')
)
df[['hh_id', 'income_per_capita', 'high_income']].head()

**Question:** Why must `high_income` reference `lambda x: x[...]` instead of `df[...]`? What is `x` at that point in the chain?

***Pergunta:** Porque é que `high_income` tem de usar `lambda x: x[...]` em vez de `df[...]`? O que é `x` nesse ponto da cadeia?*

**Answer:** Inside `assign()`, `lambda x:` receives the DataFrame **as it exists at that point in the chain**, including `income_per_capita` created earlier in the same call. Referencing `df[...]` would use the DataFrame from *before* `assign()` ran, where the new column does not yet exist.


---

## Task 4 — Multi-way categories with `np.select()`

In [ ]:
conditions = [
    df['income_dkw'] < 40000,
    df['income_dkw'] < 70000,
    df['income_dkw'] >= 70000,
]
choices = ['Low', 'Medium', 'High']

df['income_band'] = np.select(conditions, choices, default='Unknown')
df['income_band'].value_counts()

**Questions:**

- How many households fall into `Unknown`? What do they have in common?
- Why does the **order** of the conditions matter? What would happen if `>= 70000` came first?

**Answers:**

- The `Unknown` households are exactly those with a missing `income_dkw`: every numeric comparison against `NaN` is `False`, so none of the three conditions match and they fall through to `default`.
- `np.select()` takes the **first** matching condition, so list them from most to least specific. With these non-overlapping ranges the result happens to be the same either way, but as soon as ranges overlap the order decides the label.

---

## Task 5 — Custom logic with `apply()`

In [ ]:
def classify_size(size):
    if size <= 2:
        return 'Small'
    elif size <= 5:
        return 'Medium'
    else:
        return 'Large'

df['hh_size_class'] = df['hh_size'].apply(classify_size)
df['hh_size_class'].value_counts()

In [ ]:
df['high_income_flag'] = df['income_dkw'].apply(lambda x: 'Yes' if x > 50000 else 'No')
df[['hh_id', 'income_dkw', 'high_income_flag']].head()

In [ ]:
# A named function is clearer than a lambda once the logic has guards.
# apply(axis=1) passes one ROW at a time, so the function can read several columns.
def compute_income_per_capita(row):
    """Income per household member, or NaN if inputs are invalid."""
    income, hh_size = row['income_dkw'], row['hh_size']
    if pd.isna(income) or pd.isna(hh_size) or hh_size <= 0:
        return np.nan
    return round(income / hh_size, 2)

df['per_capita_safe'] = df.apply(compute_income_per_capita, axis=1)
df[['hh_id', 'income_dkw', 'hh_size', 'per_capita_safe']].head()

**Reusable cleaning functions + `apply`**

When a *new* messy extract arrives, you can wrap the cleaning rules from 2.2 in a reusable function and `apply()` it — this is the natural home for the `clean_income` and `standardise_date` helpers. The pipeline data is already clean by this stage, so we demonstrate them on sample values.

In [ ]:
def clean_income(value):
    """Strip formatting and invalid codes from a raw income string; return a float or NaN."""
    if pd.isna(value):
        return np.nan
    text = str(value)
    for ch in [' ', 'Ar', ',']:
        text = text.replace(ch, '')
    if text in ['unknown', 'NA', 'not recorded', '']:
        return np.nan
    return pd.to_numeric(text, errors='raise')

sample_income = pd.Series(['Ar 32,000', '45 000', 'unknown', '1,200,000', np.nan])
sample_income.apply(clean_income)

In [ ]:
def standardise_date(value):
    """Normalise common survey date formats to a datetime, or NaT."""
    if pd.isna(value) or str(value).strip() == "not recorded":
        return pd.NaT
    s = str(value).strip()

    # Slash formats -> ISO. "03/15/2025" is MM/DD/YYYY; "2025/01/18" is YYYY/MM/DD.
    if "/" in s:
        parts = s.split("/")
        if len(parts[0]) == 2:
            month, day, year = parts
        else:
            year, month, day = parts
        s = f"{year}-{month}-{day}"

    # Inverted month: "YYYY-DD-MM" where the middle part is > 12 -> swap day and month.
    parts = s.split("-")
    if len(parts) == 3 and int(parts[1]) > 12:
        year, day, month = parts
        s = f"{year}-{month}-{day}"

    return pd.to_datetime(s, errors="raise")

sample_dates = pd.Series(["03/15/2025", "2025/01/18", "2025-13-01", "not recorded", "2025-01-10"])
sample_dates.apply(standardise_date)

**Questions:**

- When is a **named function** better than a **lambda**? When is the lambda fine?
- The lambda flag treats `NaN > 50000` as `False` -> `'No'`. Is silently labelling unknown income as "not high" safe? How would you guard against it?

**Answers:**

- Use a **named function** when the logic spans several lines, needs a docstring, or is reused — like `classify_size`. A **lambda** is fine for a short, one-off expression such as the income flag.
- Treating `NaN > 50000` as `'No'` hides missing income inside a "not high" label. Guard it explicitly, e.g. `lambda x: 'Unknown' if pd.isna(x) else ('Yes' if x > 50000 else 'No')`, so missingness stays visible.

---

## Task 6 — Save the feature table to `20_processed/`

In [ ]:
DATA_PROC_DIR = '../../data/20_processed'
os.makedirs(DATA_PROC_DIR, exist_ok=True)
out_path = os.path.join(DATA_PROC_DIR, 'datania_households_features.csv')

df.to_csv(out_path, index=False)
print('Saved:', out_path, '|', df.shape)

In [ ]:
check = pd.read_csv(out_path, dtype={'hh_id': str, 'region_code': str})
print('Reloaded:', check.shape)
check.head()

**Questions:**

- How many columns did you add compared with the cleaned input?
- Which of your new columns are **vectorised** (fast) and which used `apply()` (slower)? On a million-row file, which would you rewrite first?

**Answers:**

- We added these derived columns: `income_per_capita`, `age_group`, `education_label`, `area_type`, `hh_category`, `income_thousands`, `pop_density_log`, `high_income`, `income_band`, `hh_size_class`, `high_income_flag`, `per_capita_safe`.
- The `np.where()`, `pd.cut()`, `map()`, `np.select()`, and direct arithmetic columns are vectorised. The `apply()` columns (`hh_size_class`, `high_income_flag`, `per_capita_safe`) iterate row-by-row; on a million-row file you would rewrite those first — for example `per_capita_safe` is just `df['income_dkw'] / df['hh_size'].where(df['hh_size'] > 0)`.